# 00 - Setup checks and data preparation (CPU only, no GPU quota)

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Get the code: preferably from the private Kaggle dataset "advanced-genai-pa2-code"
# (attached as input); otherwise clone the private GitHub repo using a Kaggle secret
# named GITHUB_TOKEN (Add-ons -> Secrets).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import glob, os, shutil, subprocess, sys
if not os.path.exists(REPO):
    code = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
            if "advanced-genai-pa2-code" in f]
    if code:
        shutil.copytree(code[0], REPO)
        print("code copied from", code[0])
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
        del token, authed
os.chdir(REPO)
print(open("CODE_VERSION.txt").read() if os.path.exists("CODE_VERSION.txt") else
      subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/

In [ ]:
!python scripts/prepare_data.py
!wc -l data/processed/*.jsonl
!cat data/processed/data_stats.json